# CANDY Tutorial: Conversation Analysis with the CANDOR Corpus

## Introduction

This notebook walks through the CANDY pipeline end-to-end using the [CANDOR corpus](https://betterup-data-resources.s3.amazonaws.com/candor-dataset/README.html) (Reece et al., 2023) — a large, naturalistic dataset of dyadic conversations paired with self-report outcomes covering affect, personality, and social connection.

The tutorial covers three sequential stages:

1. **Data Ingestion** — convert raw CANDOR transcripts into a ConvoKit-compatible corpus using `CandorConverter`.
2. **Feature Extraction** — compute conversation dynamics features (speaking time, turn length, speech rate, backchannels, response time) for each speaker using `ConversationDynamicsTransformer`.
3. **Modeling** — associate extracted features with self-reported affect change and Big-5 personality traits using an Actor-Partner Interdependence Model (APIM) with mixed effects.

The first two stages are demonstrated interactively on a 10-conversation sample. The descriptive statistics and modeling sections then work with pre-computed features from the full dataset (1,572 conversations).

### Setup

We begin with setting up the notebook with the necessary Python libraries, paths to directories, and finally a global theme for the plots.

In [12]:
import os
import random
from pathlib import Path
from glob import glob

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import plotnine as p9

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Set random seed for reproducibility
# why 42? because 42 is the answer to life, the universe and everything (The Hitchhiker's Guide to the Galaxy)
random.seed(42)

# Please change this path to your own data directory
CONVERSATIONS = "data/conversations"

p9.theme_set(
    p9.theme_minimal() + 
    p9.theme(
        figure_size=(10, 5),
        panel_background = p9.element_rect(fill='white'),
        plot_background = p9.element_rect(fill='white'),
        plot_title=p9.element_text(color='black', size=16, weight='bold'),
        axis_line=p9.element_line(color='black'),
        axis_title=p9.element_text(color='black', size=14),
        axis_text=p9.element_text(color='black', size=12),
        axis_ticks=p9.element_line(color='black')
    )
)

### Architecture

CANDY ingests conversation data in the form of transcripts, processes it through a modular pipeline involving dataset-specific conversions and dataset-agnostic transformations, and finally associates them with conversational or relational outcomes (Fig. 1).

#### 1. Converters

In the first stage, idiosyncratic conversation data (called a "corpus") is parsed and converted into a canonical representation in terms of conversations, speakers, and utterances. An easy-to-extend template is available in `candy/converters/base.py`; an off-the-shelf implementation for the CANDOR corpus is in `candy/converters/candor.py`.

#### 2. Transformers

Corpus transformations are implemented as composable stages that mutate the corpus and attach structured representations back to it. For example, `ConversationDynamicsTransformer` computes conversation dynamics features directly from utterance-level transcripts.

#### 3. Models

In this tutorial we associate the extracted features with conversational outcomes (affect change, Big-5 personality) using a mixed-effects Actor-Partner Interdependence Model (APIM).

CANDY is accessible and extensible: all core components ship off-the-shelf for common use cases while providing templates for custom converters, transformers, and models. We also provide a CLI that exposes the canonical workflows — converting raw datasets, extracting features, and exporting results — and supports both end-to-end runs and targeted re-runs on already-converted corpora.

## Step 1: Data Ingestion

We demonstrate the pipeline with a 10-conversation sample drawn from the CANDOR corpus (Reece et al., 2023). CANDOR contains 1,656 dyadic conversations collected via video call, each paired with pre- and post-conversation surveys on affect, personality, and social outcomes.

### 1.1 Converting the corpus

`CandorConverter` parses the raw CANDOR transcript files and writes a ConvoKit-compatible corpus to disk. We use the `audiophile` transcript variant, which provides word-level timing and speaker alignment.

In [4]:
# converts the CANDOR corpus to ConvoKit format
# Link to original tutorial - https://convokit.cornell.edu/documentation/candor.html
from candy.converters import CandorConverter

converter = CandorConverter(
    datapath=CONVERSATIONS,
    transcript_type="audiophile"
)

folder_name = converter.to_convokit()
print(f"Converted CANDOR corpus to ConvoKit format at folder: {CONVERSATIONS}/{folder_name}")

Converted CANDOR corpus to ConvoKit format at folder: data/conversations/candor_audiophile


The corpus is written to the directory above. Each ConvoKit corpus consists of four JSON files (conversations, speakers, utterances, corpus-level metadata) that can be inspected directly.

In [5]:
folder_name = "candor_audiophile"
!ls -lh {CONVERSATIONS}/{folder_name}

total 600K
-rw-rw-r-- 1 ssubrahmanya ssubrahmanya  22K Feb 23 18:23 conversations.json
-rw-rw-r-- 1 ssubrahmanya ssubrahmanya    2 Feb 23 18:23 corpus.json
-rw-rw-r-- 1 ssubrahmanya ssubrahmanya  18K Feb 23 18:23 index.json
-rw-rw-r-- 1 ssubrahmanya ssubrahmanya 125K Feb 23 18:23 speakers.json
-rw-rw-r-- 1 ssubrahmanya ssubrahmanya 422K Feb 23 18:23 utterances.jsonl


### 1.2 Loading the corpus

We load the converted corpus into memory as a `Corpus` object.

In [6]:
from convokit import Corpus

# load the converted corpus
corpus = Corpus(filename=Path(CONVERSATIONS) / folder_name)

## Step 2: Feature Extraction

Inspired by Di Stasi et al. (2024), we compute the following conversation dynamics features across five core dimensions. Each dimension captures a distinct aspect of how speakers interact over the course of a conversation.

- **Speaking Time:** Share of total speaking time occupied by each speaker.

- **Turn Length:** Duration of each speech turn.
  - *Metrics:* Median, mean, coefficient of variation (CV), lag-1 Spearman autocorrelation (predictability), and cross-speaker Spearman correlation with the partner's preceding turn (adaptability).

- **Speech Rate:** Words spoken per minute.
  - *Metrics:* Median, CV, predictability, adaptability.

- **Backchannels:** Short acknowledgments or feedback tokens (e.g., "mm-hmm", "okay") that overlap with the partner's turn and last under one second.
  - *Metrics:* Total turns, backchannel count, backchannel proportion.

- **Response Time:** Gap between the end of one speaker's turn and the start of the other's response, measured on non-overlapping cross-speaker transitions.
  - *Metrics:* Median, CV, predictability, adaptability.

Implementations live in `candy/transformers/conversation_dynamics/`. The `ConversationDynamicsTransformer` accepts any subset of these metrics via `register_metrics()`.

In [10]:
from candy.transformers import ConversationDynamicsTransformer

dynamics_extractor = ConversationDynamicsTransformer()
dynamics_extractor.register_metrics([
    "speaking_time",
    "turn_length",
    "speaker_rate",
    "backchannels",
    "response_time",
])

corpus = dynamics_extractor.transform(corpus)

Extracting feature: speaking_time
Extracting feature: turn_length
Extracting feature: pauses
Extracting feature: speaker_rate
Extracting feature: backchannels
Extracting feature: response_time


### 2.1 Inspecting extracted features

Features are stored in each conversation's metadata under the key `"conversation_dynamics_features"` as a nested dict: `metric → {feature_key: value}`. Speaker IDs from the original dataset are used as keys within each metric.

In [ ]:
random_conversation = corpus.random_conversation()
random_conversation.retrieve_meta("conversation_dynamics_features")

### 2.2 Exporting to CSV

`ConversationDynamicsTransformer.export()` serializes all features to a long-format CSV (one row per feature per conversation), normalizing speaker IDs to generic `speaker.0`, `speaker.1` labels so that columns align consistently across conversations.

```python
from pathlib import Path
dynamics_extractor.export(corpus, output_path=Path("results"))
# → results/conversation_dynamics_features.csv
```

The descriptive statistics and modeling sections below work with pre-computed features from the full CANDOR dataset (1,572 conversations), extracted by running the same pipeline on all conversations.

## Step 3: Descriptive Statistics

Summary statistics (M, SD, Min, Max) of the conversation dynamics features across all 1,572 CANDOR conversations. Speaker-level variables (e.g. speaking share, median turn length) are pooled across both speakers per conversation; conversation-level variables (adaptability, predictability) are one observation per conversation.

In [4]:
features = pd.read_csv("results/conversation_dynamics_features.csv")
features["stat"] = features["feature"].apply(lambda f: f.split(".")[-1] if "speaker" in f else f)
features.groupby(["metric", "stat"])["value"].describe()

count         mean         std         min  \
metric        stat                                                           
backchannels  backchannel_n      60.0    60.483333   45.296946   14.000000   
              backchannel_prop   60.0     0.234809    0.074184    0.093137   
              turns_total        60.0   242.816667  124.178452   80.000000   
response_time adaptability       60.0     0.112801    0.160134   -0.166735   
              cv                 60.0     1.507102    1.389596    0.603793   
              mean               60.0     0.786864    0.451754    0.264300   
              median             60.0     0.556133    0.253357    0.131500   
              predictability     60.0    -0.031858    0.168971   -0.556379   
speaker_rate  adaptability       60.0     0.144450    0.096920   -0.080941   
              cv                 60.0     0.942384    0.256321    0.512447   
              mean               60.0   227.613166   28.077751  179.396697   
              median             60.0   188.605661   24.710895  127.988878   
              predictability     60.0     0.073583    0.086673   -0.110737   
speaking_time share              60.0     0.500000    0.106260    0.262632   
              total_duration     60.0  1197.922433  455.545001  454.701000   
turn_length   adaptability       60.0     0.228126    0.130484   -0.042092   
              cv                 60.0     1.700314    0.950615    1.065759   
              mean               60.0     5.562526    2.472086    2.254497   
              median             60.0     2.209408    1.209361    0.461000   
              predictability     60.0     0.164725    0.096648   -0.030135   

                                       25%          50%          75%  \
metric        stat                                                     
backchannels  backchannel_n      26.750000    54.000000    77.000000   
              backchannel_prop    0.175927     0.238315     0.294588   
              turns_total       182.750000   214.000000   287.250000   
response_time adaptability        0.023553     0.114163     0.180238   
              cv                  0.865614     1.028525     1.381296   
              mean                0.549813     0.647032     0.858279   
              median              0.383750     0.516000     0.658625   
              predictability     -0.159855    -0.016634     0.100673   
speaker_rate  adaptability        0.087566     0.144262     0.197908   
              cv                  0.794685     0.914092     1.058091   
              mean              204.994551   227.062246   248.232309   
              median            175.606982   190.507916   208.123228   
              predictability      0.010207     0.078183     0.136582   
speaking_time share               0.437764     0.500000     0.562236   
              total_duration    910.574000  1152.075000  1304.600250   
turn_length   adaptability        0.145864     0.217691     0.288549   
              cv                  1.309794     1.405128     1.568410   
              mean                4.080284     4.794844     6.283144   
              median              1.202000     2.043250     3.044000   
              predictability      0.100611     0.161150     0.213634   

                                        max  
metric        stat                           
backchannels  backchannel_n      252.000000  
              backchannel_prop     0.411552  
              turns_total        722.000000  
response_time adaptability         0.534432  
              cv                   9.599604  
              mean                 3.201571  
              median               1.382000  
              predictability       0.328183  
speaker_rate  adaptability         0.418415  
              cv                   1.782138  
              mean               298.714929  
              median             234.440413  
              predictability       0.256387  
speaking_time share                0.737

## Step 4: Modeling

We associate conversation dynamics features with two sets of self-reported outcomes using an **Actor-Partner Interdependence Model (APIM)**:

- **Affect change** (`change_valence`, `change_arousal`): difference between post- and pre-conversation ratings of valence and arousal.
- **Big-5 personality** (extraversion, agreeableness, conscientiousness, neuroticism, openness): rated post-conversation.

In the APIM each observation is one speaker within one conversation. The outcome is regressed on:
- the speaker's own dynamics features (*actor effects*), and
- their conversation partner's features (*partner effects*),

with a random intercept per conversation to account for dyadic non-independence. All predictors are z-scored within each outcome's estimation sample. We use maximum-likelihood (REML=False) to support model comparison, with a fallback sequence of optimizers for numerical stability.

### 4.1 Preparing the modeling dataset

We load the pre-computed feature CSV, pivot to wide format (one row per speaker per conversation), and join with survey data to derive affect-change outcomes and Big-5 ratings. Speaker identities are aligned between the features and survey using a sort-based index.

In [13]:
features = pd.read_csv("results/audiophile/conversation_dynamics_features.csv")
parts = features["feature"].str.split('.', expand=True)
features["speaker"] = parts[1].astype(int)
features["sub"] = parts[2]
features["metric"] = features["metric"] + "." + features["sub"]

features_wide = features \
    .pivot_table(
        index=["conversation_id", "speaker"],
        columns="metric", 
        values="value",
        aggfunc="first"
    ) \
    .reset_index()

In [14]:
survey = pd \
    .read_csv(
        "data/candor_transcripts/merged_survey.csv",
        dtype={
            "user_id": str,
            "partner_id": str,
            "convo_id": str
        }
    ) \
    .drop_duplicates(subset=["convo_id", "user_id"]) \
    .reset_index(drop=True) \
    .rename(columns={"convo_id": "conversation_id"})

survey["change_valence"] = survey["affect"]  - survey["pre_affect"]
survey["change_arousal"] = survey["arousal"] - survey["pre_arousal"]

In [15]:
speakers_map = survey[["conversation_id", "user_id"]] \
    .sort_values(["conversation_id", "user_id"])

speakers_map["speaker"] = np.arange(speakers_map.shape[0]) % 2
features_wide = pd.merge(features_wide, speakers_map, on=["conversation_id", "speaker"], how="inner")

In [16]:
keep = [
    "conversation_id", "user_id", "partner_id",
    "change_valence", "change_arousal",
    "my_extraversion", "my_agreeable", "my_conscientious", "my_neurotic", "my_open",
]

apim = survey[keep] \
    .merge(features_wide, on=["conversation_id", "user_id"], how="inner")

partnered_vars = [
    "speaking_time.share",
    "turn_length.median", "turn_length.cv",
    "turn_length.adaptability", "turn_length.predictability",
    "speaker_rate.median", "speaker_rate.cv",
    "speaker_rate.adaptability", "speaker_rate.predictability",
    "backchannels.turns_total", "backchannels.backchannel_n", "backchannels.backchannel_prop",
    "response_time.median", "response_time.cv",
    "response_time.adaptability", "response_time.predictability",
]

partner = apim[["conversation_id", "user_id"] + partnered_vars] \
    .rename(columns={
        "user_id": "partner_id",
        **{v: f"partner.{v}" for v in partnered_vars},
    })

all_vars = partnered_vars + [f"partner.{v}" for v in partnered_vars]

apim = apim \
    .merge(partner, on=["conversation_id", "partner_id"], how="left") \
    .dropna(subset=all_vars) \
    .reset_index(drop=True)

for v in all_vars:
    apim[f"{v}.z"] = (apim[v] - apim[v].mean()) / apim[v].std(ddof=1)

print(f"apim: {len(apim)} rows ({apim['conversation_id'].nunique()} conversations)")
apim.head()

apim: 3144 rows (1572 conversations)


,conversation_id,user_id,partner_id,change_valence,change_arousal,my_extraversion,my_agreeable,my_conscientious,my_neurotic,my_open,...,partner.speaker_rate.cv.z,partner.speaker_rate.adaptability.z,partner.speaker_rate.predictability.z,partner.backchannels.turns_total.z,partner.backchannels.backchannel_n.z,partner.backchannels.backchannel_prop.z,partner.response_time.median.z,partner.response_time.cv.z,partner.response_time.adaptability.z,partner.response_time.predictability.z
0,4035f17c-61f5-4976-9b10-c682def66b36,5d0fb996286e1700010de2c1,5f0e50c3a8d93519188a7525,-1.0,2.0,4.000000,2.666667,3.333333,3.333333,3.000000,...,-1.397730,-2.541895,-2.376192,-1.646391,-1.306234,-0.658139,2.050899,-0.455426,0.086575,-1.934286
1,4035f17c-61f5-4976-9b10-c682def66b36,5f0e50c3a8d93519188a7525,5d0fb996286e1700010de2c1,1.0,0.0,3.000000,2.000000,3.666667,3.333333,4.333333,...,-0.659611,0.534121,-0.244868,-1.646391,-1.754663,-2.768512,1.250796,-0.392885,-0.744465,0.242300
2,46561ffd-6a0b-4958-b158-8e1d3cad91e2,5ebbfc3729e14e0372002deb,5f070f6f0b00e807015e68d4,-2.0,-1.0,3.666667,4.666667,3.666667,1.333333,4.000000,...,2.369213,0.757949,-0.238295,-0.134280,-0.462762,-0.674888,1.170786,-0.512667,-0.922360,-1.156842
3,46561ffd-6a0b-4958-b158-8e1d3cad91e2,5f070f6f0b00e807015e68d4,5ebbfc3729e14e0372002deb,0.0,2.0,3.000000,3.666667,2.000000,3.666667,2.666667,...,-0.321274,0.153746,-0.174655,-0.127927,-0.014334,0.230194,1.970889,-0.613901,-0.069888,0.348685
4,a1ab2d0e-1546-414b-945e-a59bad758e77,5dd5db1ded7ea4574c7bc996,5eac6e70e481222bfb2a8872,0.0,0.0,3.666667,3.666667,3.000000,2.333333,3.000000,...,0.436714,0.092709,-0.002507,-0.159694,-0.388024,-0.491696,-0.109378,-0.199184,1.229729,0.535526


### 4.2 Affect outcomes

We fit two mixed-effects models (valence change and arousal change) using all dynamics predictors. Each model includes a random intercept per conversation. Coefficients are reported alongside standard errors and p-values for both actor and partner effects.

In [17]:
predictors = ["speaking_time.share.z"] + [
    f"{prefix}{v}.z"
    for v in partnered_vars if v != "speaking_time.share"
    for prefix in ("", "partner.")
]
rhs = " + ".join(f"Q('{p}')" for p in predictors)


def fit_mixed(outcome, data):
    sub = data.dropna(subset=[outcome])
    formula = f"{outcome} ~ {rhs}"
    last_err = None
    for method in ("powell", "bfgs", "lbfgs", "cg", "nm"):
        try:
            return smf \
                .mixedlm(formula, sub, groups=sub["conversation_id"]) \
                .fit(reml=False, method=method)
        except np.linalg.LinAlgError as e:
            last_err = e
    raise RuntimeError(f"all optimizers failed for {outcome}: {last_err}")


def coef_table(res):
    est = "fe_params" if hasattr(res, "fe_params") else "params"
    se = "bse_fe" if hasattr(res, "bse_fe") else "bse"
    out = pd.DataFrame({"est": getattr(res, est), "se": getattr(res, se), "p": res.pvalues})
    keys = [f"Q('{p}')" for p in predictors]
    out = out.loc[[k for k in keys if k in out.index]]
    out.index = [k.removeprefix("Q('").removesuffix("')") for k in out.index]
    return out


res_valence = fit_mixed("change_valence", apim)
res_arousal = fit_mixed("change_arousal", apim)

affect_table = pd.concat(
    {"valence": coef_table(res_valence), "arousal": coef_table(res_arousal)},
    axis=1,
)

print(f"change_valence: n={int(res_valence.nobs)}, "
      f"convos={apim.dropna(subset=['change_valence'])['conversation_id'].nunique()}, "
      f"convo-var={float(res_valence.cov_re.iloc[0, 0]):.3f}")
print(f"change_arousal: n={int(res_arousal.nobs)}, "
      f"convos={apim.dropna(subset=['change_arousal'])['conversation_id'].nunique()}, "
      f"convo-var={float(res_arousal.cov_re.iloc[0, 0]):.3f}")
affect_table.round(3)

change_valence: n=3090, convos=1570, convo-var=0.253
change_arousal: n=3090, convos=1570, convo-var=0.029


valence               arousal         \
                                            est     se      p     est     se   
speaking_time.share.z                     0.041  0.045  0.362  -0.059  0.056   
turn_length.median.z                      0.080  0.057  0.162   0.121  0.068   
partner.turn_length.median.z              0.028  0.057  0.626  -0.026  0.068   
turn_length.cv.z                         -0.066  0.031  0.032  -0.030  0.037   
partner.turn_length.cv.z                  0.016  0.031  0.595  -0.072  0.037   
turn_length.adaptability.z               -0.076  0.051  0.133   0.053  0.062   
partner.turn_length.adaptability.z        0.070  0.051  0.169  -0.020  0.062   
turn_length.predictability.z              0.024  0.038  0.524   0.047  0.046   
partner.turn_length.predictability.z      0.005  0.038  0.895   0.010  0.046   
speaker_rate.median.z                     0.009  0.034  0.788   0.003  0.041   
partner.speaker_rate.median.z            -0.044  0.034  0.201  -0.023  0.041   
speaker_rate.cv.z                         0.017  0.038  0.645  -0.015  0.045   
partner.speaker_rate.cv.z                -0.013  0.037  0.733  -0.006  0.044   
speaker_rate.adaptability.z               0.041  0.029  0.164  -0.024  0.035   
partner.speaker_rate.adaptability.z       0.023  0.029  0.443  -0.029  0.035   
speaker_rate.predictability.z            -0.035  0.030  0.241  -0.055  0.036   
partner.speaker_rate.predictability.z    -0.011  0.030  0.717  -0.017  0.036   
backchannels.turns_total.z               -6.485  6.451  0.315 -12.846  8.069   
partner.backchannels.turns_total.z        6.932  6.450  0.283  13.556  8.069   
backchannels.backchannel_n.z             -0.094  0.149  0.529  -0.521  0.176   
partner.backchannels.backchannel_n.z     -0.242  0.150  0.108  -0.321  0.177   
backchannels.backchannel_prop.z           0.066  0.103  0.521   0.287  0.122   
partner.backchannels.backchannel_prop.z   0.040  0.103  0.697   0.152  0.122   
response_time.median.z                   -0.061  0.032  0.058  -0.102  0.038   
partner.response_time.median.z           -0.102  0.032  0.001  -0.097  0.038   
response_time.cv.z                        0.024  0.028  0.396   0.001  0.034   
partner.response_time.cv.z                0.007  0.028  0.790  -0.028  0.033   
response_time.adaptability.z              0.003  0.027  0.917  -0.006  0.032   
partner.response_time.adaptability.z     -0.006  0.027  0.827  -0.046  0.032   
response_time.predictability.z            0.023  0.027  0.400  -0.012  0.032   
partner.response_time.predictability.z    0.022  0.027  0.414   0.012  0.032   

                                                
                                             p  
speaking_time.share.z                    0.295  
turn_length.median.z                     0.076  
partner.turn_length.median.z             0.706  
turn_length.cv.z                         0.411  
partner.turn_length.cv.z                 0.048  
turn_length.adaptability.z               0.393  
partner.turn_length.adaptability.z       0.741  
turn_length.predictability.z             0.300  
partner.turn_length.predictability.z     0.834  
speaker_rate.median.z                    0.946  
partner.speaker_rate.median.z            0.565  
speaker_rate.cv.z                        0.739  
partner.speaker_rate.cv.z                0.894  
speaker_rate.adaptability.z              0.501  
partner.speaker_rate.adaptability.z      0.420  
speaker_rate.predictability.z            0.121  
partner.speaker_rate.predictability.z    0.630  
backchannels.turns_total.z               0.111  
partner.backchannels.turns_total.z       0.093  
backchannels.backchannel_n.z             0.003  
partner.backchannels.backchannel_n.z     0.070  
backchannels.backchannel_prop.z          0.019  
partner.backchannels.backchannel_prop.z  0.213  
response_time.median.z                   0.008  
partner.response_time.median.z           0.012  
response_time.cv.z                       0.973  
partner.response_time.cv.z 

### 4.3 Big-5 personality outcomes

We fit one model per personality trait. Mixed-effects models (with random conversation intercept) are the primary specification; OLS estimates are shown alongside for comparison. Personality ratings are treated as continuous outcomes.

In [18]:
personality = ["my_extraversion", "my_agreeable", "my_conscientious", "my_neurotic", "my_open"]

big5 = {p: fit_mixed(p, apim) for p in personality}

big5_table = pd.concat(
    {p.replace("my_", ""): coef_table(big5[p]) for p in personality},
    axis=1,
)

for p, res in big5.items():
    convos = apim.dropna(subset=[p])["conversation_id"].nunique()
    print(f"{p}: n={int(res.nobs)}, convos={convos}, "
          f"convo-var={float(res.cov_re.iloc[0, 0]):.3f}")

big5_table.round(3)

my_extraversion: n=3089, convos=1570, convo-var=0.052
my_agreeable: n=3089, convos=1570, convo-var=0.000
my_conscientious: n=3089, convos=1570, convo-var=0.000
my_neurotic: n=3089, convos=1570, convo-var=0.021
my_open: n=3089, convos=1570, convo-var=0.003


extraversion               agreeable  \
                                                 est     se      p       est   
speaking_time.share.z                          0.194  0.029  0.000    -0.019   
turn_length.median.z                           0.011  0.036  0.762     0.037   
partner.turn_length.median.z                   0.123  0.035  0.001     0.050   
turn_length.cv.z                              -0.042  0.019  0.030    -0.006   
partner.turn_length.cv.z                       0.056  0.019  0.004    -0.033   
turn_length.adaptability.z                     0.012  0.032  0.711     0.022   
partner.turn_length.adaptability.z            -0.023  0.032  0.476    -0.040   
turn_length.predictability.z                   0.018  0.024  0.441     0.000   
partner.turn_length.predictability.z           0.029  0.024  0.223    -0.006   
speaker_rate.median.z                          0.101  0.021  0.000    -0.096   
partner.speaker_rate.median.z                 -0.009  0.021  0.668    -0.005   
speaker_rate.cv.z                              0.001  0.023  0.963    -0.033   
partner.speaker_rate.cv.z                      0.029  0.023  0.219     0.004   
speaker_rate.adaptability.z                   -0.009  0.018  0.632    -0.005   
partner.speaker_rate.adaptability.z           -0.021  0.018  0.252     0.007   
speaker_rate.predictability.z                  0.014  0.019  0.456    -0.035   
partner.speaker_rate.predictability.z          0.035  0.019  0.062    -0.021   
backchannels.turns_total.z                     7.606  4.106  0.064     0.756   
partner.backchannels.turns_total.z            -7.491  4.106  0.068    -0.548   
backchannels.backchannel_n.z                  -0.125  0.092  0.176    -0.149   
partner.backchannels.backchannel_n.z          -0.035  0.093  0.705    -0.118   
backchannels.backchannel_prop.z                0.063  0.064  0.323     0.155   
partner.backchannels.backchannel_prop.z        0.154  0.064  0.016     0.173   
response_time.median.z                         0.016  0.020  0.412    -0.006   
partner.response_time.median.z                -0.025  0.020  0.207    -0.013   
response_time.cv.z                             0.008  0.018  0.667     0.000   
partner.response_time.cv.z                    -0.006  0.017  0.740     0.018   
response_time.adaptability.z                   0.017  0.017  0.316    -0.007   
partner.response_time.adaptability.z           0.019  0.017  0.269    -0.028   
response_time.predictability.z                -0.007  0.017  0.692    -0.008   
partner.response_time.predictability.z        -0.024  0.017  0.156     0.009   

                                                      conscientious         \
                                            se      p           est     se   
speaking_time.share.z                    0.025  0.451        -0.052  0.031   
turn_length.median.z                     0.030  0.221        -0.006  0.037   
partner.turn_length.median.z             0.030  0.096         0.061  0.037   
turn_length.cv.z                         0.016  0.728         0.036  0.020   
partner.turn_length.cv.z                 0.016  0.044        -0.019  0.020   
turn_length.adaptability.z               0.027  0.419        -0.046  0.033   
partner.turn_length.adaptability.z       0.027  0.148         0.040  0.034   
turn_length.predictability.z             0.020  0.989        -0.013  0.025   
partner.turn_length.predictability.z     0.020  0.777        -0.010  0.025   
speaker_rate.median.z                    0.018  0.000         0.010  0.022   
partner.speaker_rate.median.z            0.018  0.780        -0.037  0.022   
speaker_rate.cv.z                        0.020  0.097         0.006  0.024   
partner.speaker_rate.cv.z                0.020  0.825        -0.024  0.024   
speaker_rate.adaptability.z              0.016  0.764        -0.013  0.019   
partner.speaker_rate.adaptability.z      0.016  0.659        -0.007  0.019   
speaker_rate.predictability.z            0.016  0.027        -0.025  0.01

In [19]:
def fit_ols(outcome, data):
    return smf.ols(f"{outcome} ~ {rhs}", data=data.dropna(subset=[outcome])).fit()


big5_ols = {p: fit_ols(p, apim) for p in personality}

big5_ols_table = pd.concat(
    {p.replace("my_", ""): coef_table(big5_ols[p]) for p in personality},
    axis=1,
)

for p, res in big5_ols.items():
    print(f"{p}: n={int(res.nobs)}, R^2={res.rsquared:.3f}")

big5_ols_table.round(3)

my_extraversion: n=3089, R^2=0.078
my_agreeable: n=3089, R^2=0.024
my_conscientious: n=3089, R^2=0.027
my_neurotic: n=3089, R^2=0.018
my_open: n=3089, R^2=0.023


extraversion               agreeable  \
                                                 est     se      p       est   
speaking_time.share.z                          0.194  0.030  0.000    -0.019   
turn_length.median.z                           0.011  0.036  0.763     0.037   
partner.turn_length.median.z                   0.123  0.036  0.001     0.050   
turn_length.cv.z                              -0.042  0.019  0.030    -0.006   
partner.turn_length.cv.z                       0.055  0.019  0.004    -0.033   
turn_length.adaptability.z                     0.012  0.033  0.719     0.022   
partner.turn_length.adaptability.z            -0.023  0.033  0.482    -0.040   
turn_length.predictability.z                   0.019  0.024  0.438     0.000   
partner.turn_length.predictability.z           0.029  0.024  0.233    -0.006   
speaker_rate.median.z                          0.100  0.021  0.000    -0.096   
partner.speaker_rate.median.z                 -0.009  0.021  0.671    -0.005   
speaker_rate.cv.z                              0.001  0.023  0.967    -0.033   
partner.speaker_rate.cv.z                      0.029  0.023  0.217     0.004   
speaker_rate.adaptability.z                   -0.009  0.019  0.631    -0.005   
partner.speaker_rate.adaptability.z           -0.021  0.019  0.258     0.007   
speaker_rate.predictability.z                  0.014  0.019  0.447    -0.035   
partner.speaker_rate.predictability.z          0.034  0.019  0.065    -0.021   
backchannels.turns_total.z                     7.590  4.254  0.075     0.756   
partner.backchannels.turns_total.z            -7.474  4.254  0.079    -0.548   
backchannels.backchannel_n.z                  -0.125  0.092  0.176    -0.149   
partner.backchannels.backchannel_n.z          -0.036  0.093  0.698    -0.118   
backchannels.backchannel_prop.z                0.063  0.064  0.324     0.155   
partner.backchannels.backchannel_prop.z        0.155  0.064  0.016     0.173   
response_time.median.z                         0.016  0.020  0.416    -0.006   
partner.response_time.median.z                -0.025  0.020  0.217    -0.013   
response_time.cv.z                             0.007  0.018  0.692     0.000   
partner.response_time.cv.z                    -0.006  0.017  0.743     0.018   
response_time.adaptability.z                   0.017  0.017  0.318    -0.007   
partner.response_time.adaptability.z           0.019  0.017  0.275    -0.028   
response_time.predictability.z                -0.007  0.017  0.696    -0.008   
partner.response_time.predictability.z        -0.024  0.017  0.159     0.009   

                                                      conscientious         \
                                            se      p           est     se   
speaking_time.share.z                    0.025  0.454        -0.052  0.031   
turn_length.median.z                     0.030  0.224        -0.006  0.037   
partner.turn_length.median.z             0.030  0.097         0.061  0.037   
turn_length.cv.z                         0.016  0.729         0.036  0.020   
partner.turn_length.cv.z                 0.016  0.045        -0.019  0.020   
turn_length.adaptability.z               0.027  0.422        -0.046  0.034   
partner.turn_length.adaptability.z       0.028  0.150         0.040  0.034   
turn_length.predictability.z             0.020  0.989        -0.013  0.025   
partner.turn_length.predictability.z     0.020  0.778        -0.010  0.025   
speaker_rate.median.z                    0.018  0.000         0.010  0.022   
partner.speaker_rate.median.z            0.018  0.781        -0.037  0.022   
speaker_rate.cv.z                        0.020  0.098         0.006  0.024   
partner.speaker_rate.cv.z                0.020  0.826        -0.024  0.024   
speaker_rate.adaptability.z              0.016  0.765        -0.013  0.019   
partner.speaker_rate.adaptability.z      0.016  0.660        -0.007  0.019   
speaker_rate.predictability.z            0.016  0.028        -0.025  0.01